In [1]:
import geopandas as gpd
import pandas as pd
import folium

In [2]:
berk_parcels = gpd.read_parquet("../data/processed/berk_parcels.parquet")
oak_parcels = gpd.read_parquet("../data/processed/oak_scaled_parcels.parquet")
sf_parcels = gpd.read_parquet("../data/processed/sf_sb79_parcels_scaled.parquet")

In [3]:
import getpass
import os

# A laptop with the key exported as an environment variable. On DataHub this is None.
CENSUS_API_KEY = os.environ.get('CENSUS_API_KEY')

if CENSUS_API_KEY is None:
    try:
        # DataHub: a masked prompt appears below this cell. Nothing you type is saved.
        entered = getpass.getpass('Census API key (press Enter to skip): ')
        CENSUS_API_KEY = entered.strip() or None
    except Exception:
        # No interactive prompt available, which is what the autograder looks like.
        CENSUS_API_KEY = None

print('Key available:', CENSUS_API_KEY is not None)

Key available: True


In [4]:
import json
import urllib.request

acs_tables = [
    # Race & Ethnicity (B03002)
    'B03002_001E', 'B03002_003E', 'B03002_004E', 'B03002_006E', 'B03002_012E',
    # Median Household Income (B19013)
    'B19013_001E',
    # Household Income Brackets (B19001: Total + all 16 brackets)
    'B19001_001E',
    'B19001_002E', 'B19001_003E', 'B19001_004E', 'B19001_005E',
    'B19001_006E', 'B19001_007E', 'B19001_008E', 'B19001_009E',
    'B19001_010E', 'B19001_011E', 'B19001_012E', 'B19001_013E',
    'B19001_014E', 'B19001_015E', 'B19001_016E', 'B19001_017E',
    # Housing Tenure (B25003)
    'B25003_001E', 'B25003_002E', 'B25003_003E',
    # Median Home Value (B25077)
    'B25077_001E',
    # Tenure by Vehicles Available (B25044)
    'B25044_001E', 'B25044_002E', 'B25044_003E', 'B25044_009E', 'B25044_010E',
    # Commute Mode / Means of Transportation to Work (B08301)
    'B08301_001E', 'B08301_003E', 'B08301_010E', 'B08301_018E', 'B08301_019E', 'B08301_021E',
]

def fetch_census_data(api_key, geography: str, county_fips: str) -> pd.DataFrame:
    """Fetch 2024 ACS 5-year population and median household income for Alameda tracts.

    Every column comes back as text, because the geography codes are labels and the
    leading zeros have to survive. See Part I.
    """

    url = (
        'https://api.census.gov/data/2024/acs/acs5'
        f'?get=NAME,{",".join(acs_tables)}'
        f'&for={geography.replace(' ', '%20')}:*&in=state:06&in=county:{county_fips}'
        f'&key={api_key}'
    )
    with urllib.request.urlopen(url, timeout=30) as response:
        content_type = response.headers.get('Content-Type', '')
        body = response.read()

    # A missing or rejected key does not produce an HTTP error. The API sends back an
    # ordinary web page, so check what arrived before trying to read it as data.
    if 'json' not in content_type:
        raise RuntimeError('the API returned a web page instead of data; '
                           'the key was probably missing or rejected')

    # The data is a list of lists, and the first list is the header row.
    rows = json.loads(body)
    return pd.DataFrame(rows[1:], columns=rows[0])

def load_acs_data(county_fips: str) -> pd.DataFrame:
    """Load ACS data from the API or a local file."""
    api_data = None

    if CENSUS_API_KEY:
        try:
            api_data = fetch_census_data(CENSUS_API_KEY, geography='block group', county_fips=county_fips)
            print('Rows from the API:', len(api_data))
            print(api_data.head())
            return api_data
        except Exception as error:
            print('The API call did not succeed:', error)
            print('Carrying on with the file that ships with this notebook.')
    else:
        print('No key, so no API call. The shipped file is already loaded as `tracts`.')

alameda_api_acs = load_acs_data('001') # Alameda County
sf_api_acs = load_acs_data('075') # San Francisco County

Rows from the API: 1134
                                                NAME B03002_001E B03002_003E  \
0  Block Group 1; Census Tract 4001; Alameda Coun...        2074        1530   
1  Block Group 2; Census Tract 4001; Alameda Coun...        1058         521   
2  Block Group 1; Census Tract 4002; Alameda Coun...         842         584   
3  Block Group 2; Census Tract 4002; Alameda Coun...        1361         844   
4  Block Group 1; Census Tract 4003; Alameda Coun...        1233         845   

  B03002_004E B03002_006E B03002_012E B19013_001E B19001_001E B19001_002E  \
0          43         306          97      250001         791          25   
1         113         238         117      209286         546          58   
2           0          79          63      210375         393           5   
3          45         195         139      198750         486           9   
4          37         133          64      202500         536           0   

  B19001_003E  ... B08301_001E B

In [5]:
def add_geoid_column_names (api_acs: pd.DataFrame) -> pd.DataFrame:
    acs = api_acs.copy()

    #12-digit Block Group GEOID (e.g., "060014229001")
    #keep as string so the leading "0" on California ("06") is never dropped
    acs['GEOID'] = acs['state'] + acs['county'] + acs['tract'] + acs['block group']

    #convert ACS variable columns from string to numeric & clean sentinel codes
    for col in acs_tables:
        acs[col] = pd.to_numeric(acs[col], errors='coerce')
        acs.loc[acs[col] < 0, col] = pd.NA

    # Rename columns for clarity
    acs_clean = acs.rename(columns={
        # Race & Ethnicity (B03002)
        'B03002_001E': 'pop_total',
        'B03002_003E': 'pop_nh_white', #nh is not hispanic
        'B03002_004E': 'pop_nh_black',
        'B03002_006E': 'pop_nh_asian',
    'B03002_012E': 'pop_hispanic',

    # Median Household Income (B19013)
    'B19013_001E': 'median_hh_income',

    # Household Income Brackets (B19001)
    'B19001_001E': 'hh_total',
    'B19001_002E': 'hh_inc_under_10k',
    'B19001_003E': 'hh_inc_10k_to_14k',
    'B19001_004E': 'hh_inc_15k_to_19k',
    'B19001_005E': 'hh_inc_20k_to_24k',
    'B19001_006E': 'hh_inc_25k_to_29k',
    'B19001_007E': 'hh_inc_30k_to_34k',
    'B19001_008E': 'hh_inc_35k_to_39k',
    'B19001_009E': 'hh_inc_40k_to_44k',
    'B19001_010E': 'hh_inc_45k_to_49k',
    'B19001_011E': 'hh_inc_50k_to_59k',
    'B19001_012E': 'hh_inc_60k_to_74k',
    'B19001_013E': 'hh_inc_75k_to_99k',
    'B19001_014E': 'hh_inc_100k_to_124k',
    'B19001_015E': 'hh_inc_125k_to_149k',
    'B19001_016E': 'hh_inc_150k_to_199k',
    'B19001_017E': 'hh_inc_200k_plus',

    # Median Home Value (B25077)
    'B25077_001E': 'median_home_value',

    # Housing Tenure (B25003)
    'B25003_001E': 'occ_units',
    'B25003_002E': 'owner_units',
    'B25003_003E': 'renter_units',

    # Tenure by Vehicles Available (B25044)
    'B25044_001E': 'occ_units_veh_total',
    'B25044_002E': 'owner_hh_veh_total',
    'B25044_003E': 'owner_zero_car_hh',
    'B25044_009E': 'renter_hh_veh_total',
    'B25044_010E': 'renter_zero_car_hh',

    # Commute Mode (B08301)
    'B08301_001E': 'workers_total',
    'B08301_003E': 'commute_drive_alone',
    'B08301_010E': 'commute_transit',
    'B08301_018E': 'commute_bike',
    'B08301_019E': 'commute_walk',
    'B08301_021E': 'commute_wfh',
})
    return acs_clean

alameda_acs_clean = add_geoid_column_names(alameda_api_acs)
sf_acs_clean = add_geoid_column_names(sf_api_acs)


In [6]:
def summarize_acs (acs_clean: pd.DataFrame) -> pd.DataFrame:
    # 1. Compute percentage & summary columns on acs_clean using the renamed columns
    acs_clean["pct_nh_white"] = acs_clean["pop_nh_white"] / acs_clean["pop_total"]
    acs_clean["pct_nh_black"] = acs_clean["pop_nh_black"] / acs_clean["pop_total"]
    acs_clean["pct_nh_asian"] = acs_clean["pop_nh_asian"] / acs_clean["pop_total"]
    acs_clean["pct_hispanic"] = acs_clean["pop_hispanic"] / acs_clean["pop_total"]

    under_50k_cols = [
        "hh_inc_under_10k", "hh_inc_10k_to_14k", "hh_inc_15k_to_19k",
        "hh_inc_20k_to_24k", "hh_inc_25k_to_29k", "hh_inc_30k_to_34k",
        "hh_inc_35k_to_39k", "hh_inc_40k_to_44k", "hh_inc_45k_to_49k",
    ]
    under_100k_cols = under_50k_cols + [
        "hh_inc_50k_to_59k", "hh_inc_60k_to_74k", "hh_inc_75k_to_99k",
    ]

    acs_clean["hh_under_50k"] = acs_clean[under_50k_cols].sum(axis=1)
    acs_clean["hh_under_100k"] = acs_clean[under_100k_cols].sum(axis=1)
    acs_clean["pct_hh_under_50k"] = acs_clean["hh_under_50k"] / acs_clean["hh_total"]
    acs_clean["pct_hh_under_100k"] = acs_clean["hh_under_100k"] / acs_clean["hh_total"]

    acs_clean["pct_owner"] = acs_clean["owner_units"] / acs_clean["occ_units"]
    acs_clean["pct_renter"] = acs_clean["renter_units"] / acs_clean["occ_units"]

    acs_clean["zero_car_hh"] = acs_clean["owner_zero_car_hh"] + acs_clean["renter_zero_car_hh"]
    acs_clean["pct_zero_car_all"] = acs_clean["zero_car_hh"] / acs_clean["occ_units_veh_total"]
    acs_clean["pct_zero_car_owner"] = acs_clean["owner_zero_car_hh"] / acs_clean["owner_hh_veh_total"]
    acs_clean["pct_zero_car_renter"] = acs_clean["renter_zero_car_hh"] / acs_clean["renter_hh_veh_total"]

    acs_clean["pct_drive_alone"] = acs_clean["commute_drive_alone"] / acs_clean["workers_total"]
    acs_clean["pct_transit"] = acs_clean["commute_transit"] / acs_clean["workers_total"]
    acs_clean["pct_bike"] = acs_clean["commute_bike"] / acs_clean["workers_total"]
    acs_clean["pct_walk"] = acs_clean["commute_walk"] / acs_clean["workers_total"]
    acs_clean["pct_wfh"] = acs_clean["commute_wfh"] / acs_clean["workers_total"]

    return acs_clean

alameda_acs_summarized = summarize_acs(alameda_acs_clean)
sf_acs_summarized = summarize_acs(sf_acs_clean)

In [7]:
def attach_acs_to_parcels(parcels_gdf: gpd.GeoDataFrame, acs: pd.DataFrame, county_fips: str) -> gpd.GeoDataFrame:
    # Read 2024 California Block Group boundaries directly from Census TIGER/Line
    tiger_url = "https://www2.census.gov/geo/tiger/TIGER2024/BG/tl_2024_06_bg.zip"
    ca_bgs = gpd.read_file(tiger_url)

    # Filter to the specified county and keep only GEOID + geometry
    county_bgs = ca_bgs[ca_bgs["COUNTYFP"] == f'{county_fips}'][["GEOID", "geometry"]].copy()

    # Match CRS (coordinate reference system) between parcels and Census boundaries
    if parcels_gdf.crs != county_bgs.crs:
        county_bgs = county_bgs.to_crs(parcels_gdf.crs)

    # Convert parcel polygons to interior points so edge-bordering parcels match 1-to-1
    parcel_pts = parcels_gdf.copy()
    parcel_pts["geometry"] = parcel_pts.geometry.representative_point()

    # 4. Spatial join: attach 12-digit block group GEOID to every parcel
    parcels_with_bg = gpd.sjoin(
        parcel_pts,
        county_bgs,
        how="left",
        predicate="within",
    ).drop(columns=["index_right"])

    # Restore original parcel polygon geometries (if you plan to map parcels later)
    parcels_with_bg["geometry"] = parcels_gdf.geometry

    # 5. Merge your cleaned ACS DataFrame (`acs`) onto the parcels GeoDataFrame
    parcels_acs = parcels_with_bg.merge(acs, on="GEOID", how="left")

    print("Unmatched parcels:", parcels_acs["GEOID"].isna().sum())
    return parcels_acs

In [8]:
berk_parcels_acs = attach_acs_to_parcels(berk_parcels, alameda_acs_summarized, "001")
oak_parcels_acs = attach_acs_to_parcels(oak_parcels, alameda_acs_summarized, "001")
sf_parcels_acs = attach_acs_to_parcels(sf_parcels, sf_acs_summarized, "075") 

#one parcel in SF no longer exists according to SF zoning data, so it is unmatched
#one parcel in SF was on the border, so went unmatched

Unmatched parcels: 0
Unmatched parcels: 0
Unmatched parcels: 2


In [9]:
berk_parcels_acs.to_parquet("../data/processed/berk_parcels_acs.parquet")
oak_parcels_acs.to_parquet("../data/processed/oak_parcels_acs.parquet")
sf_parcels_acs.to_parquet("../data/processed/sf_parcels_acs.parquet")